In [0]:
%run /Workspace/Users/senoom222@gmail.com/databricks-code-repos-senthil/Databricks_workout_2025/Calling_1_wb_to_2_wb_using_util_run/Generic_Specific_Frame

In [0]:
dbutils.widgets.text("Catalog","")
CATALOG = dbutils.widgets.get("Catalog").strip()
dbutils.widgets.text("Schema","")
SCHEMA = dbutils.widgets.get("Schema").strip()

In [0]:
%python
import json

child_output = dbutils.notebook.run("/Workspace/Users/senoom222@gmail.com/databricks-code-repos-senthil/Databricks_workout_2025/Calling_1_wb_to_2_wb_using_util_run/config",120,{"Catalog":CATALOG,"Schema" : SCHEMA})

child_dict = json.loads(child_output)
CATALOG = child_dict["Catalog"]
SCHEMA = child_dict["Schema"]
SRC = child_dict["Source"]
BRONZE = child_dict["Bronze"]
SILVER = child_dict["Silver"]
GOLD = child_dict["Gold"]
SILVERDB = child_dict["Silver Table"]
GOLDDB = child_dict["Gold Table"]


print("Returned Source Location : ",GOLD)
print("Returned Target Location : ",GOLD)

In [0]:
from pyspark.sql.window import Window

df = spark.read.format("delta").load(f"{GOLD}/logistics_gold_curated/")
w = Window.partitionBy("orgin_hub_city").orderBy(func.col("shipment_cost").desc())

top3 = df.withColumn("rank",func.dense_rank().over(w)).filter("rank<=3")

# LEAD / LAG
lag_df = df.withColumn(
    "prev_shipment_days",
    func.lag("shipment_year").over(
        Window.partitionBy("masked_staff_name").orderBy("shipment_year")
    )
)

# CUBE AGGREGATION
cube_df = df.cube("orgin_hub_city") \
    .agg(func.sum("shipment_cost").alias("total_cost"))

deltawrite(top3,f"{GOLD}/top3_drivers")
deltawrite(lag_df,f"{GOLD}/prev_shipment_days")
deltawrite(cube_df,f"{GOLD}/cube_costs")

In [0]:
analytics_df = df.orderBy(func.col("shipment_cost").desc())
deltawrite(analytics_df,f"{GOLD}/analytics_ready")